In [1]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel,RunnableBranch,RunnableLambda
import os
os.environ['OPENAI_API_KEY']='key'
os.environ['GROQ_API_KEY']='key'

In [3]:
model=ChatOpenAI(model='gpt-5.4-mini',temperature=1)
parser=StrOutputParser()
from pydantic import BaseModel,Field
from typing import Literal

In [4]:
class Feedback(BaseModel):
    sentiment:Literal['positive','negative']=Field(description='Give the sentiment of the feedback')

from langchain_core.output_parsers import PydanticOutputParser

In [5]:
parser2=PydanticOutputParser(pydantic_object=Feedback)


In [6]:
prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into postive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)

In [7]:
classifier_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

In [8]:
prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

In [12]:
branch_chain=RunnableBranch(
    (lambda x:x.sentiment=='positive',prompt2|model|parser),
    (lambda x:x.sentiment=='negetive',prompt3|model|parser),RunnableLambda(lambda x:"could not find sentiment")
)


chain=classifier_chain|branch_chain

print(chain.invoke({'feedback':'this is a beautifull phone '}))



Thank you so much for the kind words! I’m really glad to hear you had a positive experience. We appreciate your feedback and support.


In [13]:
chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOpenAI |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
